# CEG-WM runtime qualification

Frozen thin entrypoint for real runtime qualification only. Start with `smoke`; do not use it for calibration, attacks, candidate promotion, experiments, or stage migration.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/CEG-WM/runtime_qualification"
CONTENT_ROOT = "/content"
PACKAGE_DEFAULT = f"{DRIVE_ROOT}/execution_packages/current/ceg_wm_runtime_execution.zip"
PROFILE = (input("Profile [smoke]: ") or "smoke").strip()
PACKAGE_ZIP = (input(f"Execution package path [{PACKAGE_DEFAULT}]: ") or PACKAGE_DEFAULT).strip()
EXPECTED_PACKAGE_SHA256 = input("Expected package SHA-256: ").strip().lower()
REPLAY_SOURCE = input("Replay source path [none]: ").strip() or None
BOOTSTRAP = f"{DRIVE_ROOT}/bootstrap/package_schema_1/runtime_qualification_bootstrap.py"
EXPECTED_BOOTSTRAP_SHA256 = "911eb6c80777a72b8ff3cd59968597076f5a4442e9ac1762b5709e908ba07586"

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, sys, tempfile
from google.colab import userdata
bootstrap_payload = pathlib.Path(BOOTSTRAP).read_bytes()
if hashlib.sha256(bootstrap_payload).hexdigest() != EXPECTED_BOOTSTRAP_SHA256: raise RuntimeError("trusted bootstrap SHA-256 mismatch")
snapshot_root = pathlib.Path(tempfile.mkdtemp(prefix="ceg_wm_trusted_bootstrap_", dir=CONTENT_ROOT))
TRUSTED_BOOTSTRAP = snapshot_root / "runtime_qualification_bootstrap.py"
with TRUSTED_BOOTSTRAP.open("xb") as sink: sink.write(bootstrap_payload)
if hashlib.sha256(TRUSTED_BOOTSTRAP.read_bytes()).hexdigest() != EXPECTED_BOOTSTRAP_SHA256: raise RuntimeError("trusted bootstrap snapshot SHA-256 mismatch")
if PROFILE not in {"smoke", "qualification", "replay"}: raise RuntimeError("unsupported profile")
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True, capture_output=True)
if gpu.returncode != 0: raise RuntimeError("GPU_REQUIRED: nvidia-smi failed")
print({"gpu": gpu.stdout.strip(), "content_disk_free_bytes": shutil.disk_usage(CONTENT_ROOT).free})
for secret_name in ("HF_TOKEN", "CEG_WM_ROOT_KEY"):
    secret_value = userdata.get(secret_name)
    if not secret_value: raise RuntimeError(f"{secret_name} Colab Secret is required")
    os.environ[secret_name] = secret_value
EPHEMERAL_ROOT = f"{CONTENT_ROOT}/ceg_wm_runtime"

In [ ]:
command = [sys.executable, str(TRUSTED_BOOTSTRAP), "--profile", PROFILE, "--package-zip", PACKAGE_ZIP,
           "--expected-package-sha256", EXPECTED_PACKAGE_SHA256, "--ephemeral-root", EPHEMERAL_ROOT,
           "--persistent-root", DRIVE_ROOT]
if REPLAY_SOURCE: command.extend(["--replay-source", REPLAY_SOURCE])
completed = subprocess.run(command, text=True, capture_output=True, env=os.environ)
if completed.stderr: print(completed.stderr)
status = json.loads(completed.stdout.strip().splitlines()[-1])
print(json.dumps(status, indent=2, sort_keys=True))
if completed.returncode not in (0, 1, 2, 3): raise RuntimeError("unexpected bootstrap exit code")

In [ ]:
artifact_path = status.get("result_zip") or status.get("diagnostic_zip")
print({"artifact_kind": status["artifact_kind"], "profile": status["profile"], "run_status": status["run_status"], "artifact_path": artifact_path})
if completed.returncode: raise RuntimeError(f"qualification entrypoint exited {completed.returncode}; artifact preserved at {artifact_path}")